In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma    # new version
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_ollama import ChatOllama, OllamaEmbeddings
import os 
from langchain_community.document_loaders import TextLoader

/var/folders/9t/2fbrff9979b0mg7vphsy68900000gn/T/ipykernel_23783/3925217984.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# LOCAL EMBEDDINGS using Ollama (No API key needed)
embeddings = OllamaEmbeddings(model="bge-m3")

In [3]:
# 1. Load the text files from your folder
folder_path = "my_txt_files" 
documents = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        # Using utf-8 encoding is crucial for reading Arabic characters properly
        loader = TextLoader(file_path, encoding="utf-8")
        documents.extend(loader.load())

# 2. Split documents into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [4]:
# Initialize an empty Chroma database with your model settings
# (Using a clean name to completely avoid dimension conflicts)
db = Chroma(
    embedding_function=embeddings,
    collection_name="my_rag_project", #collection of all the documents for the same project. This name is saved inside the database 
    persist_directory="./my_vector_db",
    collection_metadata={"hnsw:space": "cosine"}  # <-- CRUCIAL FOR ARABIC SEMANTICS

)

In [5]:
# Add documents in small, bite-sized batches of 10 chunks
batch_size = 10
for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    db.add_documents(batch)
    print(f"Successfully processed chunks {i} to {i + len(batch)}")

print("==== Ingestion Complete! Data saved permanently to disk ====")

Successfully processed chunks 0 to 10
Successfully processed chunks 10 to 20
Successfully processed chunks 20 to 30
Successfully processed chunks 30 to 40
Successfully processed chunks 40 to 50
Successfully processed chunks 50 to 60
Successfully processed chunks 60 to 70
Successfully processed chunks 70 to 80
Successfully processed chunks 80 to 90
Successfully processed chunks 90 to 100
Successfully processed chunks 100 to 110
Successfully processed chunks 110 to 120
Successfully processed chunks 120 to 130
Successfully processed chunks 130 to 140
Successfully processed chunks 140 to 150
Successfully processed chunks 150 to 160
Successfully processed chunks 160 to 170
Successfully processed chunks 170 to 180
Successfully processed chunks 180 to 190
Successfully processed chunks 190 to 200
Successfully processed chunks 200 to 210
Successfully processed chunks 210 to 220
Successfully processed chunks 220 to 230
Successfully processed chunks 230 to 240
Successfully processed chunks 240 to

In [6]:
# Step 4: Link your local llama3.1 engine
retriever = db.as_retriever(search_kwargs={"k": 3})
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [ ]:
# Step 5: Augment the prompt and invoke

prompt = ChatPromptTemplate.from_template("""
أنت مساعد ذكي ومتخصص في تحليل النصوص العربية ومحدد جداً. مهمتك هي الإجابة على سؤال المستخدم بناءً على السياق المستخرج المقدم فقط.

شروط الإجابة:
1. يجب أن تعتمد إجابتك بالكامل على السياق المرفق أدناه.
2. لا تضف أي معلومات خارجية تماماً من خارج هذا النص.
3. يمكنك فهم المعنى المرادف لغوياً بدقة.
4. إذا لم تكن الإجابة واضحة بشكل مباشر، يجب عليك تحليل نص الكتب بدقة واستنباط الإجابة الصحيحة بناءً على الفهم والربط المنطقي بين الأفكار الواردة.
5. يجب عليك دائماً ذكر اسم الكتاب أو المصدر الذي استخرجت منه الإجابة في نهاية ردك.

السياق المستخرج:
{context}

السؤال:
{input}
""")


doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

In [ ]:
print("\n==== محرك البث الفوري جاهز تماماً! ====")
print("يمكنك البدء بطرح الأسئلة (اكتب 'خروج' للإنهاء):")
print("-" * 50)

while True:
    # 1. Continuously prompt the user for input
    user_question = input("\nYou: ")
    
    # 2. Check if the user wants to break out of the loop
    if user_question.lower() in ['exit', 'quit','خروج']:
        print("تم إغلاق البرنامج بنجاح.")
        break
        
    # 3. Skip empty submissions
    if not user_question.strip():
        continue
        
    # 4. Invoke the chain with the live user input
    try:
        response = rag_chain.invoke({"input": user_question})
        
        # 5. Extract and print the answer safely
        # Note: LangChain RAG chains usually return a dictionary. 
        # If yours returns a raw string instead, just use: print(f"AI: {response}")
        answer = response.get("answer", response) if isinstance(response, dict) else response
        print(f"AI: {answer}")
        
    except Exception as e:
        print(f"An error occurred: {e}")